# ACDT Project 1 — Movie Metascore & Worldwide Box Office
**Updated for the supplied 2022–2025 USD dataset + interactive presentation**  
Built from Group 4's original `PROJECT(1).ipynb`, while retaining the genre-filter / OLS / Random-Forest-DML / outlier-visualization goals.

**Hypotheses**  
- Association: Higher **critic Metascore (0–100)** is associated with higher worldwide box office revenue.  
- Moderation: The Metascore–revenue relationship differs by **Action, Comedy, Drama, Horror**.  
- Model: `log(Worldwide Gross) ~ Metascore + log(Budget) + Release Year + Genre` with Metascore × Genre in the moderation model.

**Important:** These are observational, mostly post-release data. No analysis in this notebook proves that critical reviews *cause* revenue changes. Metascore can be measured after box-office performance begins; unmeasured marketing, IP, distribution, and reverse causation may remain.

> **Data provenance**: This notebook uses the supplied **407-movie USD-converted** CSV (`ACDT_movies_2022_2025_budget_USD(1).csv`) and contains a byte-identical embedded backup for use in Colab. If either `ACDT_movies_2022_2025_budget_USD(1).csv` or `ACDT_movies_2022_2025_budget_USD.csv` is uploaded, it takes priority over the embedded backup. The USD currency conversion used release-year annual-average exchange rates and is approximate. Review `ACDT_budget_conversion_audit_2022_2025.csv` separately when documenting assumptions.

## 0. Setup / 실행 방법
1. Open this `.ipynb` in **Google Colab** and choose **Runtime → Run all**. The notebook already contains a **byte-identical backup of the supplied 407-row CSV**, so uploading data is optional.  
2. To explicitly use the attached file, upload `ACDT_movies_2022_2025_budget_USD(1).csv` to the Colab Files pane (`/content/`) and rerun from Section 1. The loader also accepts `ACDT_movies_2022_2025_budget_USD.csv`.  
3. Check the printed **Data source** and **sample-selection audit**: 407 input films, 406 after excluding a `$1` budget, and **385** in the four-genre regression analysis (for this specific file).  
4. Sections 3–5 run OLS, genre interactions, holdout Random Forest comparison, and 5-fold cross-fit Random-Forest DML. Section 6 contains interactive widgets (Colab/Jupyter); Section 9 exports result CSVs.  
5. The **2025 portion is incomplete**; `log(Budget)` and `log(Revenue)` require positive USD amounts. The reported regression sample omits budgets under **USD 1,000** as a documented data-quality rule.  

**Methods:** OLS with HC3 robust standard errors; joint Wald test of interactions; 80/20 holdout prediction; Random Forest; 5-fold residualization-based DML; genre-assignment and calendar-window sensitivity checks. These models do not establish causality.


In [1]:
import ast, base64, gzip, io, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
TARGET_GENRES = ['Action','Comedy','Drama','Horror']
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 30)
print('Environment ready.')

Environment ready.


## 1. Data quality / sample-selection audit
- Keep only positive USD budget and worldwide gross for log transforms.
- **Do not** strip dollar / euro / won symbols and treat the remaining figures as USD.
- Budget `$1` for *Radical Wolfe* is implausible: exclude budgets under **USD 1,000** for this analysis, retain original file unchanged, and check the source if time allows.
- Original `genres` entries are multi-label strings. For a mutually exclusive interaction model, select the **first occurrence among our four broad genres** in the source genre list. Report ambiguous multi-genre movies. This is a modeling choice, not an objective film classification.

In [2]:
# The user-provided USD-converted dataset takes priority when available.
# The embedded copy below has identical bytes to the uploaded CSV (407 movies).
CSV_FILENAMES = ('ACDT_movies_2022_2025_budget_USD(1).csv',
                 'ACDT_movies_2022_2025_budget_USD.csv')
# In Colab the Files pane stores uploads in /content/; local Jupyter uses the working folder.
CSV_PATHS = [root / fname for root in (Path.cwd(), Path('/content'), Path('/mnt/data'))
             for fname in CSV_FILENAMES]
DATA_CSV = next((path for path in CSV_PATHS if path.is_file()), None)
DATA_BASE64_GZIP = (
    '...'
)
if DATA_CSV is not None:
    df_raw = pd.read_csv(DATA_CSV, encoding='utf-8-sig')
    data_source = f'CSV file: {DATA_CSV.resolve()}'
else:
    backup_csv = gzip.decompress(base64.b64decode(DATA_BASE64_GZIP))
    df_raw = pd.read_csv(io.BytesIO(backup_csv))
    data_source = 'Embedded copy of the uploaded 2022–2025 USD dataset (no upload needed)'
print('Data source:', data_source)
print('Rows loaded:', len(df_raw), '| Columns:', len(df_raw.columns))

Data source: CSV file: /content/ACDT_movies_2022_2025_budget_USD.csv
Rows loaded: 407 | Columns: 10


In [3]:
required = ['title','metascore','rating','votes','budget','gross_worldwide',
            'gross_us_canada','release_date','genres','duration']
missing_cols = sorted(set(required) - set(df_raw.columns))
if missing_cols:
    raise ValueError(f'Missing columns: {missing_cols}. Use the USD-converted movie CSV.')

df = df_raw.copy()
for name in ['metascore','rating','votes','budget','gross_worldwide','gross_us_canada']:
    df[name] = pd.to_numeric(df[name], errors='coerce')
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year

def split_genres(value):
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except (ValueError, SyntaxError):
        pass
    return [x.strip() for x in str(value).split(',')]

def choose_primary(value):
    # Consistently pick the first genre from the original list that is one of the 4 targets.
    return next((g for g in split_genres(value) if g in TARGET_GENRES), None)

df['primary_genre'] = df['genres'].map(choose_primary)
df['target_genre_count'] = df['genres'].map(
    lambda v: sum(g in split_genres(v) for g in TARGET_GENRES))

mask = (df['release_year'].between(2022, 2025) & df['metascore'].between(0,100)
        & (df['budget'] >= 1000) & (df['gross_worldwide'] > 0) & (df['votes'] >= 100)
        & df[required].notna().all(axis=1))
df_valid = df.loc[mask].copy()
df_model = df_valid.dropna(subset=['primary_genre']).copy()
df_model['log_revenue'] = np.log(df_model['gross_worldwide'])
df_model['log_budget'] = np.log(df_model['budget'])
df_model['year_centered'] = df_model['release_year'] - 2022

audit = pd.DataFrame({
    'Stage':['Uploaded / embedded USD-converted CSV',
             'After numeric/date/sample validation',
             '4 primary target genres'],
    'Movies':[len(df),len(df_valid),len(df_model)]
})
print(audit.to_string(index=False))
print('Excluded for implausibly low budget (<$1,000):', int((df['budget']<1000).sum()))
print('Multi-label among the four genres:', int((df_model['target_genre_count']>=2).sum()))
print('Genre counts:',df_model['primary_genre'].value_counts().to_dict())
print('Year counts:',df_model['release_year'].value_counts().sort_index().to_dict())
display(df_model[['title','metascore','budget','gross_worldwide','primary_genre','release_year']].head())

                                Stage  Movies
Uploaded / embedded USD-converted CSV     407
 After numeric/date/sample validation     406
              4 primary target genres     385
Excluded for implausibly low budget (<$1,000): 1
Multi-label among the four genres: 143
Genre counts: {'Action': 125, 'Drama': 120, 'Comedy': 93, 'Horror': 47}
Year counts: {2022: 113, 2023: 123, 2024: 120, 2025: 29}


,title,metascore,budget,gross_worldwide,primary_genre,release_year
0,Beast,54,36000000,59095809,Action,2022
1,Winnie-the-Pooh: Blood and Honey 2,27,1000000,7582541,Horror,2024
2,The Crime Is Mine,74,14819290,10473857,Comedy,2023
3,The Boogeyman,55,35000000,67307243,Horror,2023
4,Beau Is Afraid,63,35000000,12337378,Comedy,2023


## 2. Descriptive evidence / first visualization
Scatter plots use **individual films**. Large earners remain visible and are **not automatically deleted**.  
Genre filtering changes what you see; regression sample remains defined consistently.

In [4]:
try:
    import plotly.express as px
    hover = ['title','budget','release_year','rating','votes']
    fig = px.scatter(df_model, x='metascore', y='gross_worldwide', color='primary_genre',
                     hover_name='title', hover_data=['budget','release_year','rating'],
                     log_y=True, opacity=0.66,
                     title='Critic Metascore vs Worldwide Box Office (log-scale USD)',
                     labels={'metascore':'Critic Metascore (0–100)',
                             'gross_worldwide':'Worldwide box office (USD, log scale)',
                             'primary_genre':'Assigned primary genre'})
    fig.update_layout(height=520)
    fig.show()
except ImportError:
    plt.figure(figsize=(9,5))
    for genre, sub in df_model.groupby('primary_genre'):
        plt.scatter(sub.metascore,sub.gross_worldwide,alpha=.5,label=genre)
    plt.yscale('log');plt.xlabel('Metascore');plt.ylabel('Worldwide box office (USD, log)')
    plt.legend();plt.show()

## 3. Hypothesis testing: baseline, multiple regression, genre interaction
**Model A:** `log(revenue) ~ metascore`  
**Model B:** `log(revenue) ~ metascore + log(budget) + C(release_year) + C(primary_genre)`  
**Model C:** `log(revenue) ~ metascore * C(primary_genre) + log(budget) + C(release_year)`

OLS p-values use **HC3 heteroskedasticity-robust** standard errors. The joint interaction Wald test asks whether the Metascore slope differs across genres. A 10-point effect in a log-outcome model corresponds to `100 * (exp(10 * beta) − 1)%`, holding modeled covariates fixed (association, not causation).

In [5]:
formula_simple = 'log_revenue ~ metascore'
formula_multiple = 'log_revenue ~ metascore + log_budget + C(release_year) + C(primary_genre)'
formula_interaction = ('log_revenue ~ metascore * C(primary_genre) + '
                       'log_budget + C(release_year)')
ols_simple = smf.ols(formula_simple, data=df_model).fit(cov_type='HC3')
ols_multiple = smf.ols(formula_multiple, data=df_model).fit(cov_type='HC3')
ols_interaction = smf.ols(formula_interaction, data=df_model).fit(cov_type='HC3')

def report_ols(model, label):
    beta = float(model.params['metascore'])
    ci = model.conf_int().loc['metascore'].to_numpy()
    return {'Model':label,'N':int(model.nobs),'R²':round(model.rsquared,3),
            'Adj R²':round(model.rsquared_adj,3),
            'Metascore β (log/$ per point)':round(beta,4),
            'Metascore p':round(float(model.pvalues['metascore']),5),
            '10-point %':round(100*np.expm1(10*beta),1),
            '10-point % 95% CI':f'[{100*np.expm1(10*ci[0]):.1f}, {100*np.expm1(10*ci[1]):.1f}]'}
comparison_ols = pd.DataFrame([report_ols(ols_simple,'A: Simple'),
                               report_ols(ols_multiple,'B: Multiple')])
display(comparison_ols)
print('MULTIPLE REGRESSION — HC3 coefficient table:')
display(ols_multiple.summary2().tables[1])

interaction_terms = [i for i,name in enumerate(ols_interaction.params.index)
                     if name.startswith('metascore:C(primary_genre)')]
if interaction_terms:
    R = np.zeros((len(interaction_terms),len(ols_interaction.params)))
    for k,i in enumerate(interaction_terms): R[k,i] = 1
    wald_p = float(ols_interaction.wald_test(R, scalar=True).pvalue)
    print(f'Joint interaction Wald test (HC3): p = {wald_p:.5g}')
else:
    wald_p = np.nan
    print('Not enough genre variation to test interaction.')

# Compute group-specific slopes as appropriate linear combinations of coefficients.
rows=[]
params_names = list(ols_interaction.params.index)
for genre in TARGET_GENRES:
    # Construct contrast vector directly from parameter names
    contrast = np.zeros(len(params_names))

    # 'metascore' is always part of the slope
    if 'metascore' in params_names:
        contrast[params_names.index('metascore')] = 1.0

    # If it's a non-reference genre, add the interaction term coefficient
    genre_interaction_name = f'metascore:C(primary_genre)[T.{genre}]'
    if genre_interaction_name in params_names:
        contrast[params_names.index(genre_interaction_name)] = 1.0

    test = ols_interaction.t_test(contrast)
    slope = float(test.effect.item())
    interval = np.asarray(test.conf_int()).ravel()
    rows.append({'Genre':genre, 'N':int((df_model.primary_genre==genre).sum()),
                 'β (log USD / 1 score)':round(slope,4),
                 'p':round(float(test.pvalue),5),
                 '10-point %':round(100*np.expm1(10*slope),1),
                 '10-point % 95% CI':f'[{100*np.expm1(10*interval[0]):.1f}, {100*np.expm1(10*interval[1]):.1f}]'})
interaction_table=pd.DataFrame(rows)
display(interaction_table)

,Model,N,R²,Adj R²,Metascore β (log/$ per point),Metascore p,10-point %,10-point % 95% CI
0,A: Simple,385,0.004,0.001,-0.0095,0.20481,-9.1,"[-21.5, 5.3]"
1,B: Multiple,385,0.582,0.573,0.0206,0.00077,22.9,"[9.0, 38.5]"


MULTIPLE REGRESSION — HC3 coefficient table:


,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,-3.774060,1.857038,-2.032301,4.212322e-02,-7.413787,-0.134332
C(release_year)[T.2023],-0.126123,0.204777,-0.615904,5.379577e-01,-0.527478,0.275232
C(release_year)[T.2024],-0.016669,0.206722,-0.080635,9.357323e-01,-0.421836,0.388498
C(release_year)[T.2025],-0.381064,0.396416,-0.961273,3.364149e-01,-1.158027,0.395898
C(primary_genre)[T.Comedy],-0.120449,0.216788,-0.555610,5.784772e-01,-0.545345,0.304446
C(primary_genre)[T.Drama],-0.570355,0.241869,-2.358114,1.836807e-02,-1.044410,-0.096300
C(primary_genre)[T.Horror],1.671962,0.280252,5.965927,2.432493e-09,1.122678,2.221245
metascore,0.020580,0.006119,3.363415,7.698455e-04,0.008587,0.032572
log_budget,1.159563,0.095906,12.090669,1.183198e-33,0.971591,1.347535


Joint interaction Wald test (HC3): p = 0.049296


,Genre,N,β (log USD / 1 score),p,10-point %,10-point % 95% CI
0,Action,125,0.0377,0.00001,45.8,"[24.0, 71.4]"
1,Comedy,93,-0.0007,0.94887,-0.7,"[-21.0, 24.6]"
2,Drama,120,0.0156,0.14927,16.8,"[-5.4, 44.4]"
3,Horror,47,0.0195,0.40961,21.5,"[-23.5, 93.0]"


## 4. Random Forest: **prediction**, not causal effect
Predict **log(revenue)** from Metascore, log budget, release year and primary genre. Both a linear baseline and the RF model are assessed on the **same held-out 20% sample**. Do not interpret feature importance as causality. Prediction errors on log revenue are not expressed as $ errors.

In [6]:
features=['metascore','log_budget','release_year','primary_genre']
X=df_model[features].copy()
y=df_model['log_revenue'].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.20,random_state=RANDOM_STATE)

def make_pipeline(est):
    return Pipeline([
        ('prep', ColumnTransformer([
            ('num','passthrough',['metascore','log_budget','release_year']),
            ('genre',OneHotEncoder(handle_unknown='ignore'),['primary_genre'])])),
        ('model',est)
    ])
linear_predictor=make_pipeline(LinearRegression())
rf_predictor=make_pipeline(RandomForestRegressor(
    n_estimators=250,min_samples_leaf=3,random_state=RANDOM_STATE,n_jobs=-1))
results=[]
for name,est in [('Linear baseline',linear_predictor),('Random Forest',rf_predictor)]:
    est.fit(X_train,y_train)
    yp=est.predict(X_test)
    results.append({'Model':name,'Test N':len(y_test),
                    'Test R² (log revenue)':round(r2_score(y_test,yp),3),
                    'Test MAE (log USD)':round(mean_absolute_error(y_test,yp),3)})
rf_results=pd.DataFrame(results)
display(rf_results)

permutation=permutation_importance(rf_predictor,X_test,y_test,
                        n_repeats=10,scoring='neg_mean_absolute_error',random_state=42)
importance=pd.DataFrame({'Feature':X_test.columns,
                         'Permutation importance (change in log-MAE)':permutation.importances_mean,
                         'SD':permutation.importances_std}).sort_values(
                         'Permutation importance (change in log-MAE)',ascending=False)
display(importance)
print('Caution: Feature importance is predictive, not causal; with a small test set it can be unstable.')

,Model,Test N,Test R² (log revenue),Test MAE (log USD)
0,Linear baseline,77,0.511,1.148
1,Random Forest,77,0.564,1.152


,Feature,Permutation importance (change in log-MAE),SD
1,log_budget,1.000655,0.134440
3,primary_genre,0.228276,0.042410
2,release_year,-0.009188,0.013456
0,metascore,-0.026244,0.045249


Caution: Feature importance is predictive, not causal; with a small test set it can be unstable.


## 5. Double Machine Learning — manual cross-fitted PLR
Following the **partialling-out** idea from the Week 5 DML example:
1. Treat Metascore as continuous `D` and `log(revenue)` as outcome `Y`.
2. Treat `log(budget)`, year, genre indicators as observed controls `W`.
3. Use random forests with **5-fold cross-fitting** to predict `E[D|W]` and `E[Y|W]` using data outside each observation's fold.
4. Regress the **out-of-fold residualized outcome** on the **out-of-fold residualized Metascore** with an orthogonal moment score and asymptotic SE.

We do **not** include `votes` as a primary control: vote counts are measured after release, can be affected by popularity/revenue, and may introduce post-treatment bias.  
The result is an **adjusted association** unless the untestable causal identification assumptions (no unmeasured confounding, timing, valid model, etc.) hold.

In [7]:
W=pd.concat([
    df_model[['log_budget','release_year']].reset_index(drop=True),
    pd.get_dummies(df_model['primary_genre'].reset_index(drop=True),
                   prefix='genre',drop_first=True,dtype=float)
],axis=1).astype(float)
D=df_model['metascore'].to_numpy(dtype=float)
Y=df_model['log_revenue'].to_numpy(dtype=float)

fold=KFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
rD=np.full(len(D),np.nan)
rY=np.full(len(Y),np.nan)
for fold_i,(train_i,test_i) in enumerate(fold.split(W)):
    m=RandomForestRegressor(n_estimators=200,min_samples_leaf=5,
                            random_state=RANDOM_STATE+fold_i,n_jobs=-1)
    g=RandomForestRegressor(n_estimators=200,min_samples_leaf=5,
                            random_state=RANDOM_STATE+100+fold_i,n_jobs=-1)
    m.fit(W.iloc[train_i],D[train_i])
    g.fit(W.iloc[train_i],Y[train_i])
    rD[test_i]=D[test_i]-m.predict(W.iloc[test_i])
    rY[test_i]=Y[test_i]-g.predict(W.iloc[test_i])
if not (np.isfinite(rD).all() and np.isfinite(rY).all()):
    raise ValueError('DML cross-fitted residuals contain missing values.')

# Orthogonal PLR score psi_i(theta)=rD_i*(rY_i-theta*rD_i).
dml_beta=float(np.sum(rD*rY)/np.sum(rD**2))
psi = rD*(rY-dml_beta*rD)
psi_a_mean = np.mean(-rD**2)
dml_se=float(np.sqrt(np.mean(psi**2)/psi_a_mean**2/len(D)))
dml_ci=(dml_beta-1.96*dml_se,dml_beta+1.96*dml_se)
dml_p=float(2*stats.norm.sf(abs(dml_beta/dml_se)))
dml_results=pd.DataFrame([{
    'Method':'5-fold RF-DML (PLR, partialling out)',
    'N':len(D),'Metascore β (log USD / point)':dml_beta,
    'SE (orthogonal score)':dml_se,'p (approx)':dml_p,
    'CI low':dml_ci[0],'CI high':dml_ci[1],
    '10-point %':100*np.expm1(10*dml_beta)
}])
display(dml_results.round(5))
print('DML is NOT proof of a causal effect; assumptions may be violated in movie data.')

# Optional: check the official Week 5 API if you install `doubleml` in Colab.
# %pip -q install doubleml
# from doubleml import DoubleMLData, DoubleMLPLR
# full = pd.concat([df_model[['log_revenue','metascore']].reset_index(drop=True),W],axis=1)
# dml_obj = DoubleMLData(full,y_col='log_revenue',d_cols='metascore',x_cols=list(W.columns))
# fit = DoubleMLPLR(dml_obj,
#        ml_l=RandomForestRegressor(n_estimators=200,min_samples_leaf=5,random_state=42),
#        ml_m=RandomForestRegressor(n_estimators=200,min_samples_leaf=5,random_state=42),
#        n_folds=5,score='partialling out').fit()
# display(fit.summary)

,Method,N,Metascore β (log USD / point),SE (orthogonal score),p (approx),CI low,CI high,10-point %
0,"5-fold RF-DML (PLR, partialling out)",385,0.02176,0.00554,0.00009,0.0109,0.03261,24.30765


DML is NOT proof of a causal effect; assumptions may be violated in movie data.


## 5B. Confidence-interval visualization / uncertainty comparison
The coefficient plots show **adjusted associations** and 95% confidence intervals. The **interaction model** differs by genre; its estimates should not be described as separate randomized experiments. Point estimates close to zero and intervals spanning zero need cautious interpretation. DML and OLS use different adjustment methods and do not automatically identify a causal effect.

In [8]:
# Robust 95% confidence-interval comparison of pooled MLR and manual DML.
try:
    import plotly.graph_objects as go
    beta_mlr = float(ols_multiple.params['metascore'])
    ci_mlr = ols_multiple.conf_int().loc['metascore'].to_numpy()
    effect_df = pd.DataFrame([
        {'Method':'Multiple OLS (HC3)', 'estimate':beta_mlr, 'lower':ci_mlr[0], 'upper':ci_mlr[1]},
        {'Method':'Random-Forest DML', 'estimate':dml_beta, 'lower':dml_ci[0], 'upper':dml_ci[1]}
    ])
    fig_ci=go.Figure()
    for _,r in effect_df.iterrows():
        fig_ci.add_trace(go.Scatter(x=[r.estimate],y=[r.Method],mode='markers',
                marker={'size':11},error_x={'type':'data','symmetric':False,
                'array':[r.upper-r.estimate], 'arrayminus':[r.estimate-r.lower]},
                name=r.Method))
    fig_ci.add_vline(x=0,line_dash='dash',line_color='gray')
    fig_ci.update_layout(title='Metascore coefficient: adjusted associations & 95% CIs',
        xaxis_title='Coefficient per Metascore point (ln worldwide USD)',
        yaxis_title='',height=340,showlegend=False)
    fig_ci.show()

    genre_ci=interaction_table.copy()
    # Use original full-precision values computed robustly from parameter names for the correct plotted interval.
    genre_points=[]
    params_names = list(ols_interaction.params.index)
    for genre in TARGET_GENRES:
        # Construct contrast vector directly from parameter names
        contrast = np.zeros(len(params_names))

        # 'metascore' is always part of the slope
        if 'metascore' in params_names:
            contrast[params_names.index('metascore')] = 1.0

        # If it's a non-reference genre, add the interaction term coefficient
        genre_interaction_name = f'metascore:C(primary_genre)[T.{genre}]'
        if genre_interaction_name in params_names:
            contrast[params_names.index(genre_interaction_name)] = 1.0

        tst = ols_interaction.t_test(contrast)
        cc = np.asarray(tst.conf_int()).ravel()
        genre_points.append((genre, float(tst.effect.item()), cc[0], cc[1]))

    fig_g=go.Figure()
    for genre,mu,lo,hi in genre_points:
        fig_g.add_trace(go.Scatter(x=[mu],y=[genre],mode='markers',name=genre,
             marker={'size':10},error_x={'type':'data','symmetric':False,
             'array':[hi-mu],'arrayminus':[mu-lo]}))
    fig_g.add_vline(x=0,line_dash='dash',line_color='gray')
    fig_g.update_layout(title='Metascore × Genre: genre-specific slopes & 95% CIs',
        xaxis_title='Coefficient per Metascore point (ln worldwide USD)',
        height=370,showlegend=False)
    fig_g.show()
except ImportError:
    print('Optional confidence-interval charts require plotly.')

## 6. Interactive Presentation — select genre / Metascore / budget / year
**Actual fitted model:** the *genre-interaction multiple regression* (not a made-up formula). Drag sliders and press the button. Predicted dollar figures are the **exponentiated predictions from a log model** (typical/geometric scale), **not unbiased forecasts of mean dollar revenue**; 95% intervals below describe uncertainty of the modeled conditional mean on the log scale.

The scatter also retains a **10-film Metascore-bin average** inspired by the team's original visualization. These bins are **descriptive only**, not used to fit the model. Top grossing titles are annotated, *not deleted*.

In [9]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print('ipywidgets is unavailable in this runtime. In Google Colab it is normally installed.')

# Unit-testable helper: any slider combination uses the actual fitted interaction model.
def predict_median_gross(genre='Action',metascore=70,budget_m=50,year=2024):
    if genre not in TARGET_GENRES: raise ValueError('Unknown genre')
    new = pd.DataFrame({'primary_genre':[genre],'metascore':[float(metascore)],
                        'log_budget':[np.log(float(budget_m)*1e6)],
                        'release_year':[int(year)]})
    sf=ols_interaction.get_prediction(new).summary_frame(alpha=.05).iloc[0]
    return {'USD':float(np.exp(sf['mean'])),
            'CI_low_USD':float(np.exp(sf['mean_ci_lower'])),
            'CI_high_USD':float(np.exp(sf['mean_ci_upper']))}

if HAS_WIDGETS:
    genre_widget=widgets.Dropdown(options=TARGET_GENRES,value='Action',description='Genre:')
    metascore_widget=widgets.IntSlider(value=70,min=0,max=100,step=5,
                                      description='Metascore:',continuous_update=False,
                                      style={'description_width':'initial'})
    budget_widget=widgets.FloatSlider(value=50,min=1,max=250,step=1,
                                     description='Budget ($M):',continuous_update=False,
                                     style={'description_width':'initial'})
    year_widget=widgets.Dropdown(options=[2022,2023,2024,2025],value=2024,description='Year:')
    view_widget=widgets.ToggleButtons(options=['Raw movies','10-film bins'],description='View:')
    run_widget=widgets.Button(description='Run analysis',button_style='success')
    result_output=widgets.Output()

    def on_click_run(_):
        with result_output:
            clear_output(wait=True)
            genre=genre_widget.value
            sub=df_model.loc[df_model.primary_genre==genre].sort_values('metascore').copy()
            estimates=predict_median_gross(genre,metascore_widget.value,
                                            budget_widget.value,year_widget.value)
            print(f'{genre} | Metascore {metascore_widget.value} | '
                  f'Budget ${budget_widget.value:,.0f}M | {year_widget.value}')
            print(f'Adjusted typical revenue: ${estimates["USD"]/1e6:,.1f}M')
            print(f'95% confidence interval (conditional log mean): '
                  f'${estimates["CI_low_USD"]/1e6:,.1f}M–'
                  f'${estimates["CI_high_USD"]/1e6:,.1f}M')
            plt.figure(figsize=(10,5.6))
            if view_widget.value=='Raw movies':
                plt.scatter(sub.metascore,sub.gross_worldwide/1e6,
                            alpha=.58,s=32,label=f'Individual films (n={len(sub)})')
            else:
                # Preserve team's original binning idea, but without removing outliers.
                sub['bin']=np.arange(len(sub))//10
                grp=sub.groupby('bin',as_index=False).agg(
                    metascore=('metascore','mean'),gross_worldwide=('gross_worldwide','mean'))
                plt.scatter(grp.metascore,grp.gross_worldwide/1e6,s=70,
                            alpha=.8,label='Mean of consecutive 10 movies')
            highest=sub.nlargest(3,'gross_worldwide')
            for _,row in highest.iterrows():
                plt.annotate(str(row.title)[:24],(row.metascore,row.gross_worldwide/1e6),
                             fontsize=7,xytext=(4,4),textcoords='offset points')
            xs=np.arange(0,101,2)
            preds=np.array([predict_median_gross(genre,x,budget_widget.value,
                                               year_widget.value)['USD'] for x in xs])/1e6
            plt.plot(xs,preds,color='crimson',linewidth=2.5,
                     label='Adjusted model trend (fixed budget/year)')
            plt.scatter([metascore_widget.value],[estimates['USD']/1e6],
                        marker='*',color='black',s=180,label='Selected scenario')
            plt.yscale('log')
            plt.xlim(0,100)
            plt.xlabel('Critic Metascore (0–100)')
            plt.ylabel('Worldwide box office (million USD; log scale)')
            plt.title(f'{genre}: Metascore & worldwide gross (no outlier trimming)')
            plt.legend(loc='best');plt.grid(alpha=.2)
            plt.tight_layout();plt.show()
    run_widget.on_click(on_click_run)
    display(widgets.VBox([widgets.HTML('<h3>🎬 Movie Ratings vs Revenue — Interactive Pitch</h3>'),
                          genre_widget,metascore_widget,budget_widget,year_widget,view_widget,
                          run_widget,result_output]))
    print('Click Run analysis to see the model-based prediction and chart.')

Click Run analysis to see the model-based prediction and chart.


## 7. Animated data coverage (progressive evidence)
For the pitch, this Plotly animation gradually reveals movies from **2022**, **2022–2023**, **2022–2024**, and **2022–2025**. It illustrates how the observed data coverage changes (not a causal time effect). If Plotly is unavailable, skip this optional visualization.

In [10]:
try:
    import plotly.express as px
    stages=[]
    for through in sorted(df_model.release_year.unique()):
        part=df_model[df_model.release_year<=through].copy()
        part['Data included through']=str(int(through))
        stages.append(part)
    animated=pd.concat(stages,ignore_index=True)
    animated['Movie ID']=animated['title']+' / '+animated['release_date'].dt.strftime('%Y-%m-%d')
    anim=px.scatter(animated,x='metascore',y='log_revenue',color='primary_genre',
         animation_frame='Data included through',animation_group='Movie ID',
         hover_name='title',hover_data=['budget','release_year'],
         range_x=[0,100],range_y=[float(df_model.log_revenue.min())-1,float(df_model.log_revenue.max())+1],
         title='Progressive data coverage by release year',
         labels={'log_revenue':'ln(Worldwide gross USD)','metascore':'Critic Metascore'})
    anim.update_layout(height=520)
    anim.show()
except ImportError:
    print('Install plotly if you want an animated view.')

## 8. Sensitivity & diagnostic checks
- **Year sensitivity:** 2025 is only partially covered in this downloaded dataset, so compare 2022–2024 with 2022–2025.
- **Influential films:** identify unusually influential observations with Cook's distance; **review**, don't automatically drop.
- **Selection limits:** Budget / Metascore complete-case restriction may cause nonrepresentative sample; exclusion of streaming-only, limited-release or missing-budget films could affect conclusions.

In [11]:
prev=df_model[df_model.release_year<=2024]
prev_model=smf.ols(formula_multiple,data=prev).fit(cov_type='HC3')
sensitivity=pd.DataFrame([
    report_ols(ols_multiple,'2022–2025 (partial 2025)'),
    report_ols(prev_model,'2022–2024 only')
])
display(sensitivity)

# Cook's distance from a non-robust OLS fit is an influence diagnostic, not a deletion rule.
ols_influence=smf.ols(formula_multiple,data=df_model).fit()
cook=ols_influence.get_influence().cooks_distance[0]
review=df_model[['title','metascore','gross_worldwide','budget','release_year','primary_genre']].copy()
review['Cooks_D']=cook
print('Top influential movies: review manually (not removed):')
display(review.nlargest(7,'Cooks_D'))

,Model,N,R²,Adj R²,Metascore β (log/$ per point),Metascore p,10-point %,10-point % 95% CI
0,2022–2025 (partial 2025),385,0.582,0.573,0.0206,0.00077,22.9,"[9.0, 38.5]"
1,2022–2024 only,356,0.591,0.583,0.0201,0.00166,22.2,"[7.9, 38.5]"


Top influential movies: review manually (not removed):


,title,metascore,gross_worldwide,budget,release_year,primary_genre,Cooks_D
268,Skinamarink,66,2116254,11116,2023,Horror,0.108685
343,Winnie-the-Pooh: Blood and Honey,16,7717044,100000,2023,Horror,0.082774
204,Borderline,63,13472,5000000,2025,Comedy,0.065662
181,Memory,71,2074586,100000,2024,Drama,0.038724
375,We're All Going to the World's Fair,78,105341,15000,2022,Drama,0.038528
312,Conspiracy of Silence,46,2598,3000000,2023,Drama,0.038218
342,Badland,30,25309,10000000,2023,Action,0.035894


## 9. Results for the executive summary / system maps
**Map A — statistical relationships:** include only sample-supported relationships (e.g., which coefficients and interactions are statistically distinguishable from zero). A statistically insignificant estimate is **not proof that the effect is zero**.  
**Map B — theory/scenarios:** budget, marketing intensity, theater/screen availability, brand/IP, stars, release timing, Metascore timing, genre. Unobserved variables can matter even when not in the regression.  
**Map C — ML/predictive:** annotate Random Forest permutation importance and nonlinear patterns. **Feature importance is not a causal arrow.**

**Important interpretation:** Log-outcome effects refer to relative changes on the model's conditional log scale; exponentiated model predictions are not automatically unbiased forecasts of average-dollar revenue. DML cannot eliminate unobserved confounding or reverse causation on its own.

In [12]:
# Export compact model evidence for a 1-page executive summary / slides.
from IPython.display import Markdown, display
beta=float(ols_multiple.params['metascore'])
p=float(ols_multiple.pvalues['metascore'])
rf_winner=rf_results.loc[rf_results['Test MAE (log USD)'].idxmin(),'Model']
print(f'Sample: {len(df_model)} movies, 2022–2025 (partial 2025).\n'
      f'Multiple regression: a 10-point Metascore difference is associated with '
      f'{100*np.expm1(10*beta):+.1f}% modeled worldwide gross, '
      f'conditional on budget, year and assigned primary genre (p={p:.3g}).\n'
      f'Genre-interaction joint test: p={wald_p:.3g}.\n'
      f'Lower log-scale MAE on one 20% test split: {rf_winner}.\n'
      f'DML adjusted coefficient (log USD per Metascore point): {dml_beta:+.4f} '
      f'(p≈{dml_p:.3g}).\n'
      'None of these estimates establishes causal effects.')
comparison_ols.to_csv('ACDT_OLS_comparison.csv',index=False)
interaction_table.to_csv('ACDT_genre_interaction.csv',index=False)
rf_results.to_csv('ACDT_prediction_comparison.csv',index=False)
dml_results.to_csv('ACDT_DML_results.csv',index=False)
print('Exports: ACDT_OLS_comparison.csv, ACDT_genre_interaction.csv, '
      'ACDT_prediction_comparison.csv, ACDT_DML_results.csv')

Sample: 385 movies, 2022–2025 (partial 2025).
Multiple regression: a 10-point Metascore difference is associated with +22.9% modeled worldwide gross, conditional on budget, year and assigned primary genre (p=0.00077).
Genre-interaction joint test: p=0.0493.
Lower log-scale MAE on one 20% test split: Linear baseline.
DML adjusted coefficient (log USD per Metascore point): +0.0218 (p≈8.54e-05).
None of these estimates establishes causal effects.
Exports: ACDT_OLS_comparison.csv, ACDT_genre_interaction.csv, ACDT_prediction_comparison.csv, ACDT_DML_results.csv


10. Additional robustness checks — Genre coding and 5-fold predictive validation
Why this matters: Our moderation finding is borderline (interaction p ≈ .049). About 143/385 films belong to more than one of our four genre categories, so the arbitrary first-listed genre choice could influence the result. We compare several transparent alternatives without selecting the smallest p-value.
We also add 5-fold evaluation of predictive performance. Test folds are kept separate from the training folds; every model's preprocessing is fitted only to its training fold. These are sensitivity analyses, not causal identification.
Interpretation reminder: A non-significant result in a smaller subgroup is not proof of no effect; comparing subgroup p-values is not itself a formal test of differences. Read the estimates and uncertainty together.

In [13]:
# 10A. Genre-coding / calendar-window robustness
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Use the genre counts already calculated in Section 1.
def robust_joint_genre_p(data):
    m = smf.ols(formula_interaction, data=data).fit(cov_type='HC3')
    terms = [j for j, nm in enumerate(m.params.index)
             if nm.startswith('metascore:C(primary_genre)')]
    if not terms:
        return np.nan
    r = np.zeros((len(terms), len(m.params)))
    for k, j in enumerate(terms):
        r[k, j] = 1
    return float(m.wald_test(r, scalar=True).pvalue)

samples = [
    ('Main analysis (primary genre)', df_model),
    ('2022–2024 only', df_model.loc[df_model.release_year <= 2024]),
    ('Exactly one target genre', df_model.loc[df_model.target_genre_count == 1]),
]
check_rows = []
for label, sub in samples:
    fitted = smf.ols(formula_multiple, data=sub).fit(cov_type='HC3')
    check_rows.append({
        'Sample': label,
        'N': len(sub),
        'Metascore 10-point assoc. (%)': round(100 * np.expm1(10 * fitted.params['metascore']),1),
        'Metascore p (additive)': round(float(fitted.pvalues['metascore']),4),
        'Genre interaction joint p': round(robust_joint_genre_p(sub),4)
    })

# Rather than assigning one primary genre, encode all four genres as nonexclusive binary features.
multi_label = df_model.copy()
for genre in TARGET_GENRES:
    multi_label['has_' + genre.lower()] = multi_label['genres'].map(
        lambda v, name=genre: float(name in split_genres(v)))
controls = ' + '.join('has_' + g.lower() for g in TARGET_GENRES)
formula_multi = 'log_revenue ~ metascore + log_budget + C(release_year) + ' + controls
formula_multi_int = formula_multi + ' + ' + ' + '.join(
    'metascore:has_' + g.lower() for g in TARGET_GENRES)
model_multi = smf.ols(formula_multi, data=multi_label).fit(cov_type='HC3')
model_multi_int = smf.ols(formula_multi_int, data=multi_label).fit(cov_type='HC3')
inter_names = ['metascore:has_' + g.lower() for g in TARGET_GENRES]
r = np.zeros((len(inter_names),len(model_multi_int.params)))
for k, name in enumerate(inter_names):
    r[k, list(model_multi_int.params.index).index(name)] = 1
multi_p = float(model_multi_int.wald_test(r, scalar=True).pvalue)
check_rows.append({
    'Sample':'Nonexclusive multi-label genre coding',
    'N':len(multi_label),
    'Metascore 10-point assoc. (%)':round(100*np.expm1(10*model_multi.params['metascore']),1),
    'Metascore p (additive)':round(float(model_multi.pvalues['metascore']),4),
    'Genre interaction joint p':round(multi_p,4)
})
robustness_table = pd.DataFrame(check_rows)
display(robustness_table)
print('Interpretation: genre moderation near p=.05 is fragile to the choice of genre coding / sample.')
print('The single-genre-only subset has fewer movies and thus less power.')
robustness_table.to_csv('ACDT_genre_robustness_checks.csv',index=False)

,Sample,N,Metascore 10-point assoc. (%),Metascore p (additive),Genre interaction joint p
0,Main analysis (primary genre),385,22.9,0.0008,0.0493
1,2022–2024 only,356,22.2,0.0017,0.0503
2,Exactly one target genre,242,16.8,0.0513,0.9231
3,Nonexclusive multi-label genre coding,385,24.3,0.0008,0.0479


Interpretation: genre moderation near p=.05 is fragile to the choice of genre coding / sample.
The single-genre-only subset has fewer movies and thus less power.


In [14]:
# 10B. Five-fold cross-validation on the same dataset (no re-use of test data for fitting)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
cv = KFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
cv_estimates = [
    ('Linear Regression',LinearRegression()),
    ('Random Forest',RandomForestRegressor(
        n_estimators=150,min_samples_leaf=3,random_state=RANDOM_STATE,n_jobs=-1)),
]
cv_rows=[]
for label, estimator in cv_estimates:
    pipe=Pipeline([
        ('preprocessing',ColumnTransformer([
            ('num','passthrough',['metascore','log_budget','release_year']),
            ('genre',OneHotEncoder(handle_unknown='ignore'),['primary_genre'])
        ])),
        ('model',estimator)
    ])
    r2_values = cross_val_score(pipe, X, y, scoring='r2', cv=cv)
    mae_values = -cross_val_score(pipe, X, y, scoring='neg_mean_absolute_error', cv=cv)
    cv_rows.append({'Model':label,'CV mean R²':round(r2_values.mean(),3),
        'CV R² SD':round(r2_values.std(),3),'CV mean MAE (log scale)':round(mae_values.mean(),3),
        'CV MAE SD':round(mae_values.std(),3)})
cv_table=pd.DataFrame(cv_rows)
display(cv_table)
cv_table.to_csv('ACDT_prediction_5fold_cv.csv',index=False)
print('5-fold results complement, but do not replace, the single holdout metrics in Section 4.')


,Model,CV mean R²,CV R² SD,CV mean MAE (log scale),CV MAE SD
0,Linear Regression,0.547,0.080,1.217,0.068
1,Random Forest,0.585,0.049,1.188,0.020


5-fold results complement, but do not replace, the single holdout metrics in Section 4.
